In [47]:
pip install nltk

Note: you may need to restart the kernel to use updated packages.


In [48]:
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\vinot\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\vinot\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vinot\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns 

from sklearn.model_selection import train_test_split

import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import (accuracy_score,classification_report,confusion_matrix)

In [50]:
tickets = pd.read_csv("tickets.csv")

In [51]:
tickets

,subject,body,category
0,Payment failed,My payment was declined while making a purchase,Billing
1,Refund pending,I requested a refund but have not received it,Billing
2,Wrong invoice,The amount shown on my invoice is incorrect,Billing
3,Double charged,I was charged twice for the same order,Billing
4,Payment issue,I am unable to complete my payment,Billing
...,...,...,...
934,Address not updating,My delivery address is not updating and the pa...,General
935,Promo code blocked,My promo code is blocked at checkout and I am ...,General
936,Complaint not registered,My formal complaint failed to register and my ...,General
937,App purchase failed,My in-app purchase has failed but my card was ...,General


In [52]:
tickets["category"].value_counts()

category
Billing      235
HR           235
General      235
Technical    234
Name: count, dtype: int64

In [53]:
tickets.duplicated().sum()

np.int64(29)

In [54]:
tickets.drop_duplicates(inplace=True)

In [55]:
tickets.duplicated().sum()

np.int64(0)

In [56]:
tickets["category"].value_counts()

category
Billing      235
General      226
Technical    225
HR           224
Name: count, dtype: int64

In [57]:
tickets.isna().sum()

subject     0
body        0
category    0
dtype: int64

In [58]:
tickets["text"] = tickets["subject"] + " " + tickets["body"]

In [59]:
tickets.head()

,subject,body,category,text
0,Payment failed,My payment was declined while making a purchase,Billing,Payment failed My payment was declined while m...
1,Refund pending,I requested a refund but have not received it,Billing,Refund pending I requested a refund but have n...
2,Wrong invoice,The amount shown on my invoice is incorrect,Billing,Wrong invoice The amount shown on my invoice i...
3,Double charged,I was charged twice for the same order,Billing,Double charged I was charged twice for the sam...
4,Payment issue,I am unable to complete my payment,Billing,Payment issue I am unable to complete my payment


In [60]:
tickets.tail()

,subject,body,category,text
934,Address not updating,My delivery address is not updating and the pa...,General,Address not updating My delivery address is no...
935,Promo code blocked,My promo code is blocked at checkout and I am ...,General,Promo code blocked My promo code is blocked at...
936,Complaint not registered,My formal complaint failed to register and my ...,General,Complaint not registered My formal complaint f...
937,App purchase failed,My in-app purchase has failed but my card was ...,General,App purchase failed My in-app purchase has fai...
938,VIP support blocked,My VIP support access is blocked and I am unab...,General,VIP support blocked My VIP support access is b...


Clean the Text

In [61]:


stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    return " ".join(words)

tickets['text'] = tickets['text'].apply(preprocess)

Split Dataset

In [62]:
X = tickets["text"]

In [63]:
y = tickets["category"]

In [64]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,test_size=0.2,random_state=42,stratify=y)

In [65]:
X

0        payment failed payment declined making purchase
1               refund pending requested refund received
2           wrong invoice amount shown invoice incorrect
3                     double charged charged twice order
4                  payment issue unable complete payment
                             ...                        
934    address updating delivery address updating pac...
935    promo code blocked promo code blocked checkout...
936    complaint registered formal complaint failed r...
937    app purchase failed inapp purchase failed card...
938    vip support blocked vip support access blocked...
Name: text, Length: 910, dtype: object

In [66]:
y

0      Billing
1      Billing
2      Billing
3      Billing
4      Billing
        ...   
934    General
935    General
936    General
937    General
938    General
Name: category, Length: 910, dtype: object

In [67]:
len(X_train)        # training data

728

In [68]:
len(X_test)          # testing data

182

TF-IDF

In [69]:
vectorizer = TfidfVectorizer(lowercase = True,stop_words="english",ngram_range=(1,2),max_features=5000,min_df=2,max_df=0.95,sublinear_tf=True)

X_train_tfidf = vectorizer.fit_transform(X_train)

X_test_tfidf = vectorizer.transform(X_test)

In [70]:
X_train_tfidf.shape

(728, 837)

In [71]:
X_test_tfidf.shape

(182, 837)

Model Training

Model 1 : Navie Bayes

In [72]:
model = MultinomialNB(alpha=0.5)

In [73]:
model.fit(X_train_tfidf, y_train)

,alpha,0.5
,force_alpha,True
,fit_prior,True
,class_prior,None


Use GridSearchCV

In [74]:


params = {'alpha':[0.1,0.5,1,2]}

grid = GridSearchCV(MultinomialNB(), params, cv=5)

grid.fit(X_train_tfidf, y_train)

print("Best alpha:",grid.best_params_)

Best alpha: {'alpha': 1}


In [75]:
best_model = grid.best_estimator_

In [76]:
y_pred = best_model.predict(X_test_tfidf)

In [77]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
print("Accuracy percentage:", accuracy*100, "%")

Accuracy: 0.8131868131868132
Accuracy percentage: 81.31868131868131 %


In [78]:
print(classification_report(y_test,y_pred,zero_division=0))

              precision    recall  f1-score   support

     Billing       0.79      0.94      0.85        47
     General       0.79      0.60      0.68        45
          HR       0.89      0.87      0.88        45
   Technical       0.79      0.84      0.82        45

    accuracy                           0.81       182
   macro avg       0.81      0.81      0.81       182
weighted avg       0.81      0.81      0.81       182



In [79]:
print(confusion_matrix(y_test,y_pred))

[[44  2  0  1]
 [ 6 27  4  8]
 [ 2  3 39  1]
 [ 4  2  1 38]]


In [80]:
# 5 new support tickets
new_tickets = [
    "My credit card payment was declined",
    "I cannot reset my password",
    "I want to apply for sick leave",
    "I was charged twice for my purchase",
    "Immediate technical support is required"
    
]

# Convert new tickets into TF-IDF features
new_tickets_tfidf = vectorizer.transform(new_tickets)

# Predict categories
predictions = model.predict(new_tickets_tfidf)

# Get prediction probabilities
probabilities = model.predict_proba(new_tickets_tfidf)

# Display results
print("\nNew Ticket Predictions:")

for ticket, prediction, probability in zip(
    new_tickets,
    predictions,
    probabilities
):
    confidence = max(probability) * 100

    print("\nTicket:", ticket)
    print("Predicted Category:", prediction)
    print("Confidence:", round(confidence, 2), "%")


New Ticket Predictions:

Ticket: My credit card payment was declined
Predicted Category: Billing
Confidence: 95.25 %

Ticket: I cannot reset my password
Predicted Category: Technical
Confidence: 62.75 %

Ticket: I want to apply for sick leave
Predicted Category: HR
Confidence: 80.76 %

Ticket: I was charged twice for my purchase
Predicted Category: Billing
Confidence: 74.0 %

Ticket: Immediate technical support is required
Predicted Category: General
Confidence: 61.64 %


In [81]:
# Confidence threshold
threshold = 60

print("\nFinal Results:")

for ticket, prediction, probability in zip(
    new_tickets,
    predictions,
    probabilities
):
    confidence = max(probability) * 100

    if confidence < threshold:
        status = "Needs Human Review"
    else:
        status = "Automatically Categorized"

    print("\nTicket:", ticket)
    print("Category:", prediction)
    print("Confidence:", round(confidence, 2), "%")
    print("Status:", status)


Final Results:

Ticket: My credit card payment was declined
Category: Billing
Confidence: 95.25 %
Status: Automatically Categorized

Ticket: I cannot reset my password
Category: Technical
Confidence: 62.75 %
Status: Automatically Categorized

Ticket: I want to apply for sick leave
Category: HR
Confidence: 80.76 %
Status: Automatically Categorized

Ticket: I was charged twice for my purchase
Category: Billing
Confidence: 74.0 %
Status: Automatically Categorized

Ticket: Immediate technical support is required
Category: General
Confidence: 61.64 %
Status: Automatically Categorized


In [82]:
# Function to detect ticket priority
def detect_priority(ticket):
    urgent_keywords = [
        "urgent",
        "emergency",
        "immediate",
        "as soon as possible",
        "critical",
        "not working",
        "failed",
        "blocked",
        "unable"
    ]

    ticket_lower = ticket.lower()

    for keyword in urgent_keywords:
        if keyword in ticket_lower:
            return "Urgent"

    return "Normal"

In [83]:
print("\nFinal Ticket Results:")

for ticket, prediction, probability in zip(
    new_tickets,
    predictions,
    probabilities
):
    # Calculate confidence
    confidence = max(probability) * 100

    # Human review
    if confidence < 60:
        status = "Needs Human Review"
    else:
        status = "Automatically Categorized"

    # Detect priority
    priority = detect_priority(ticket)

    print("\n-----------------------------")
    print("Ticket:", ticket)
    print("Category:", prediction)
    print("Confidence:", round(confidence, 2), "%")
    print("Priority:", priority)
    print("Status:", status)


Final Ticket Results:

-----------------------------
Ticket: My credit card payment was declined
Category: Billing
Confidence: 95.25 %
Priority: Normal
Status: Automatically Categorized

-----------------------------
Ticket: I cannot reset my password
Category: Technical
Confidence: 62.75 %
Priority: Normal
Status: Automatically Categorized

-----------------------------
Ticket: I want to apply for sick leave
Category: HR
Confidence: 80.76 %
Priority: Normal
Status: Automatically Categorized

-----------------------------
Ticket: I was charged twice for my purchase
Category: Billing
Confidence: 74.0 %
Priority: Normal
Status: Automatically Categorized

-----------------------------
Ticket: Immediate technical support is required
Category: General
Confidence: 61.64 %
Priority: Urgent
Status: Automatically Categorized


In [93]:
import joblib
joblib.dump(best_model,"ticket_model.pkl") 
joblib.dump(vectorizer,"tfidf_vectorizer.pkl")
print("Best model and TF-IDF vectorizer saved") 

Best model and TF-IDF vectorizer saved


=================================================================================================================

Model 2 : 

In [94]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

model_lisvc = LinearSVC(random_state=42)
model_lisvc.fit(X_train_tfidf, y_train)

y_pred_lisvc = model_lisvc.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred_lisvc))
print(classification_report(y_test, y_pred_lisvc))

Accuracy: 0.8241758241758241
              precision    recall  f1-score   support

     Billing       0.86      0.94      0.90        47
     General       0.76      0.62      0.68        45
          HR       0.81      0.87      0.84        45
   Technical       0.85      0.87      0.86        45

    accuracy                           0.82       182
   macro avg       0.82      0.82      0.82       182
weighted avg       0.82      0.82      0.82       182



In [95]:
print(confusion_matrix(y_test,y_pred))

[[44  2  0  1]
 [ 6 27  4  8]
 [ 2  3 39  1]
 [ 4  2  1 38]]


==========================================================================================================

Model 3 :Random Forest

In [96]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

model_rf = RandomForestClassifier(n_estimators=200, random_state=42)
model_rf.fit(X_train_tfidf, y_train)

y_pred_rf = model_rf.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

Accuracy: 0.8076923076923077
              precision    recall  f1-score   support

     Billing       0.89      0.87      0.88        47
     General       0.75      0.60      0.67        45
          HR       0.83      0.89      0.86        45
   Technical       0.75      0.87      0.80        45

    accuracy                           0.81       182
   macro avg       0.81      0.81      0.80       182
weighted avg       0.81      0.81      0.80       182



====================================================================================================================

Model 4 : XGBoost

In [97]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [98]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

model_xg = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)

model_xg.fit(X_train_tfidf, y_train_enc)

y_pred_xg = model_xg.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test_enc, y_pred_xg))

Accuracy: 0.7637362637362637


===================================================================================================================

Model 5 : Logistic Regression

In [99]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

model_loR = LogisticRegression(max_iter=1000, random_state=42)
model_loR.fit(X_train_tfidf, y_train)

y_pred_loR = model_loR.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred_loR))
print(classification_report(y_test, y_pred_loR))

Accuracy: 0.8351648351648352
              precision    recall  f1-score   support

     Billing       0.88      0.91      0.90        47
     General       0.81      0.64      0.72        45
          HR       0.87      0.87      0.87        45
   Technical       0.79      0.91      0.85        45

    accuracy                           0.84       182
   macro avg       0.83      0.83      0.83       182
weighted avg       0.84      0.84      0.83       182



In [100]:
new_tickets = [
    "My credit card payment was declined",
    "I cannot reset my password",
    "I want to apply for sick leave",
    "I was charged twice for my purchase",
    "Immediate technical support is required"
    
]

# Convert new tickets into TF-IDF features
new_tickets_tfidf = vectorizer.transform(new_tickets)

# Predict categories
predictions = model_loR.predict(new_tickets_tfidf)

# Get prediction probabilities
probabilities = model_loR.predict_proba(new_tickets_tfidf)

# Display results
print("\nNew Ticket Predictions:")

for ticket, prediction, probability in zip(
    new_tickets,
    predictions,
    probabilities
):
    confidence = max(probability) * 100

    print("\nTicket:", ticket)
    print("Predicted Category:", prediction)
    print("Confidence:", round(confidence, 2), "%")


New Ticket Predictions:

Ticket: My credit card payment was declined
Predicted Category: Billing
Confidence: 87.88 %

Ticket: I cannot reset my password
Predicted Category: Technical
Confidence: 49.35 %

Ticket: I want to apply for sick leave
Predicted Category: HR
Confidence: 75.77 %

Ticket: I was charged twice for my purchase
Predicted Category: Billing
Confidence: 55.23 %

Ticket: Immediate technical support is required
Predicted Category: General
Confidence: 61.53 %


In [101]:
# Confidence threshold
threshold = 60

print("\nFinal Results:")

for ticket, prediction, probability in zip(
    new_tickets,
    predictions,
    probabilities
):
    confidence = max(probability) * 100

    if confidence < threshold:
        status = "Needs Human Review"
    else:
        status = "Automatically Categorized"

    print("\nTicket:", ticket)
    print("Category:", prediction)
    print("Confidence:", round(confidence, 2), "%")
    print("Status:", status)


Final Results:

Ticket: My credit card payment was declined
Category: Billing
Confidence: 87.88 %
Status: Automatically Categorized

Ticket: I cannot reset my password
Category: Technical
Confidence: 49.35 %
Status: Needs Human Review

Ticket: I want to apply for sick leave
Category: HR
Confidence: 75.77 %
Status: Automatically Categorized

Ticket: I was charged twice for my purchase
Category: Billing
Confidence: 55.23 %
Status: Needs Human Review

Ticket: Immediate technical support is required
Category: General
Confidence: 61.53 %
Status: Automatically Categorized


==================================================================================================================

Finally,best performing model is Navie Bayes

========================================END=============================================